# 06. Business Recommendations

Translate the analytical findings into actionable business recommendations that can improve delivery speed, reduce late deliveries, and optimize operational planning.

### Key Business Questions

- Which factors have the greatest impact on delivery delays?
- Which operational situations require immediate intervention?
- What actions can reduce late deliveries?
- How can delivery operations be improved using the analytical findings?

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/food_delivery_cleaned.csv")

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


## 1. Overall Business Performance

Summarize the overall delivery performance to establish the business baseline before making operational recommendations.

### Key Business Questions

- What is the overall delivery performance?
- What percentage of deliveries are late?
- How large is the current delivery-time problem?
- What should be the primary business improvement objective?

In [6]:
overall_perfomance = pd.DataFrame({
    "Metric" : [
        "Total Deliveries",
        "Average Delivery Time",
        "Median Delivery Time",
        "Fastest Delivery",
        "Slowest Delivery",
        "Late Deliveries",
        "Late Delivery Rate"
    ],
    "Value" : [
        len(df),
        round(df["Time_taken(min)"].mean(),2),
        df["Time_taken(min)"].median(),
        df["Time_taken(min)"].min(),
        df["Time_taken(min)"].max(),
        (df["Time_taken(min)"] > 30).sum(),
        round((df["Time_taken(min)"] > 30).mean()*100,2)
    ]   
})
overall_perfomance

,Metric,Value
0,Total Deliveries,45593.00
1,Average Delivery Time,26.29
2,Median Delivery Time,26.00
3,Fastest Delivery,10.00
4,Slowest Delivery,54.00
5,Late Deliveries,13606.00
6,Late Delivery Rate,29.84


### Section 1 Interpretation

**1. What is the overall delivery performance?**  
The dataset contains **45,593 deliveries** with an average delivery time of **26.29 minutes** and a median of **26 minutes**.

**2. What percentage of deliveries are late?**  
There are **13,606 late deliveries**, representing **29.84%** of all deliveries. This means approximately **3 out of every 10 deliveries** exceed the 30-minute threshold.

**3. How large is the current delivery-time problem?**  
Delivery times range from **10 to 54 minutes**, showing substantial variation in delivery performance.

**4. What should be the primary business improvement objective?**  
The primary objective should be to reduce the **29.84% late-delivery rate**, while maintaining the current average delivery time of approximately 26 minutes.

### Business Insight

Late deliveries represent a significant operational challenge, with nearly **30% of all orders exceeding 30 minutes**. Reducing delays should therefore be a primary operational priority.

## 2. Traffic Management Recommendation

Analyze the impact of traffic conditions and identify actions that can reduce delays during high-traffic periods.

### Key Business Questions

- How does traffic density affect delivery performance?
- Which traffic condition creates the greatest delivery risk?
- How much higher is the late-delivery rate during Jam traffic?
- What operational action should be taken during heavy traffic?

In [10]:
df["Late_Delivery"] = np.where(
    df["Time_taken(min)"]>30,
    "Late",
    "On Time"
)

In [11]:
traffic_recommendation = (
    df.groupby("Road_traffic_density")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

traffic_recommendation["Late_Rate"] = (
    traffic_recommendation["Late_Rate"].round(2)
)

traffic_recommendation["Avg_Delivery_Time"] = (
    traffic_recommendation["Avg_Delivery_Time"].round(2)
)

traffic_recommendation

,Road_traffic_density,Orders,Avg_Delivery_Time,Late_Rate
1,Jam,14143,31.18,50.64
0,High,4425,27.24,33.56
3,Medium,10947,26.70,32.47
4,NaN,601,26.54,30.95
2,Low,15477,21.27,7.87


### Section 2 Interpretation

**1. How does traffic density affect delivery performance?**  
Delivery performance deteriorates as traffic becomes heavier. **Jam traffic** has the highest average delivery time at **31.18 minutes**, compared with only **21.27 minutes** during Low traffic.

**2. Which traffic condition creates the greatest delivery risk?**  
**Jam traffic** is the highest-risk condition, with a **50.64% late-delivery rate** across **14,143 orders**.

**3. How much higher is the late-delivery rate during Jam traffic?**  
The late-delivery rate during Jam traffic is **50.64%**, compared with only **7.87%** during Low traffic — a difference of approximately **42.77 percentage points**.

**4. What operational action should be taken?**  
During heavy traffic, the business should consider **dynamic ETA adjustments, route optimization, and reducing multiple-order assignments** to delivery partners.

### Business Insight

Traffic is one of the strongest operational drivers of delivery delays. **Jam traffic affects a large volume of orders (14,143) and results in a 50.64% late-delivery rate.** Delivery operations should therefore use traffic conditions as an important input for ETA calculation, routing, and workload allocation.

**Recommendation:** Prioritize proactive intervention during Jam traffic rather than applying the same delivery strategy across all traffic conditions.

## 3. Weather-Based Delivery Recommendation

Analyze weather conditions to identify situations where delivery delays are more likely and determine appropriate operational actions.

### Key Business Questions

- Which weather conditions create the highest delivery risk?
- How much does poor weather affect delivery time?
- Which weather conditions should trigger proactive intervention?
- What operational actions can reduce weather-related delays?

In [12]:
weather_recommendation = (
    df.groupby("Weatherconditions")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

weather_recommendation["Avg_Delivery_Time"] = (
    weather_recommendation["Avg_Delivery_Time"].round(2)
)

weather_recommendation["Late_Rate"] = (
    weather_recommendation["Late_Rate"].round(2)
)

weather_recommendation

,Weatherconditions,Orders,Avg_Delivery_Time,Late_Rate
1,conditions Fog,7654,28.92,43.79
0,conditions Cloudy,7536,28.92,43.78
5,conditions Windy,7422,26.12,26.56
2,conditions Sandstorms,7495,25.88,25.95
3,conditions Stormy,7586,25.87,25.53
4,conditions Sunny,7284,21.86,12.52


### Section 3 Interpretation

**1. Which weather conditions create the highest delivery risk?**  
**Fog and Cloudy weather** create the highest risk. Both have an average delivery time of **28.92 minutes** and late-delivery rates of approximately **43.8%**.

**2. How much does poor weather affect delivery time?**  
Deliveries during Fog or Cloudy conditions take approximately **7.06 minutes longer** on average than deliveries during Sunny conditions (**28.92 vs 21.86 minutes**).

**3. Which weather conditions should trigger proactive intervention?**  
Fog and Cloudy conditions should receive the highest priority because they affect a large number of orders and have late-delivery rates above **43%**.

**4. What operational actions can reduce weather-related delays?**  
The business should consider **proactive ETA adjustments, additional delivery capacity, and more conservative workload assignments** during poor weather conditions.

### Business Insight

Weather conditions have a clear operational impact on delivery performance. **Fog and Cloudy conditions produce the highest late-delivery rates**, while Sunny conditions have the lowest late rate at only **12.52%**.

**Recommendation:** Incorporate weather conditions into ETA calculations and delivery workload planning, particularly during Fog and Cloudy conditions.

## 4. Delivery Workload Recommendation

Analyze the effect of multiple deliveries on delivery performance and identify workload policies that can reduce delays.

### Key Business Questions

- How does delivery workload affect delivery time?
- Which workload levels have the highest late-delivery risk?
- Should multiple-order assignments be restricted?
- What workload policy should be recommended?

In [13]:
workload_recommendation = (
    df.groupby("multiple_deliveries")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

workload_recommendation["Avg_Delivery_Time"] = (
    workload_recommendation["Avg_Delivery_Time"].round(2)
)

workload_recommendation["Late_Rate"] = (
    workload_recommendation["Late_Rate"].round(2)
)

workload_recommendation

,multiple_deliveries,Orders,Avg_Delivery_Time,Late_Rate
2,2,1985,40.45,100.00
3,3,361,47.82,100.00
1,1,28159,26.86,30.26
4,NaN,993,22.77,18.63
0,0,14095,22.88,18.12


### Section 4 Interpretation

**1. How does delivery workload affect delivery time?**  
Delivery time increases substantially as the number of assigned deliveries increases. Orders with **2 deliveries** take an average of **40.45 minutes**, while orders with **3 deliveries** take **47.82 minutes**. In comparison, orders with 0 deliveries average only **22.88 minutes**.

**2. Which workload levels have the highest late-delivery risk?**  
Orders involving **2 or 3 deliveries have a 100% late-delivery rate**. These groups include **1,985 and 361 orders**, respectively, making the pattern operationally important.

**3. Should multiple-order assignments be restricted?**  
Yes, particularly during difficult operating conditions. The results indicate that assigning **2+ deliveries to the same delivery partner substantially increases delivery time and late-delivery risk**.

**4. What workload policy should be recommended?**  
The business should **limit 2+ delivery assignments**, especially during **Jam traffic or poor weather**, and prioritize single-order assignments when operational conditions are challenging.

### Business Insight

Multiple-delivery workload is a major operational risk. Deliveries involving **2+ orders show extremely high historical delay rates**, with average delivery times exceeding 40 minutes.

**Recommendation:** Use workload limits and avoid assigning multiple orders during high-risk conditions. This can help reduce excessive delivery times and improve on-time performance.

## 5. Weather × Traffic Combined Recommendation

Analyze the combined effect of weather and traffic conditions to identify the most critical operating situations and determine where proactive intervention is required.

### Key Business Questions

- Which weather and traffic combinations create the highest delivery risk?
- How much does the combination of poor weather and heavy traffic increase delays?
- Which combinations should trigger proactive operational intervention?
- What actions can reduce delays under these conditions?

In [14]:
weather_traffic_recommendation = (
    df.groupby(
        ["Road_traffic_density", "Weatherconditions"]
    )
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

weather_traffic_recommendation["Avg_Delivery_Time"] = (
    weather_traffic_recommendation["Avg_Delivery_Time"].round(2)
)

weather_traffic_recommendation["Late_Rate"] = (
    weather_traffic_recommendation["Late_Rate"].round(2)
)

weather_traffic_recommendation.head(10)

,Road_traffic_density,Weatherconditions,Orders,Avg_Delivery_Time,Late_Rate
7,Jam,conditions Fog,2429,36.81,76.08
6,Jam,conditions Cloudy,2349,36.69,74.84
18,Medium,conditions Cloudy,1838,28.48,48.31
19,Medium,conditions Fog,1852,28.04,45.90
11,Jam,conditions Windy,2351,30.22,45.56
8,Jam,conditions Sandstorms,2399,30.02,44.52
9,Jam,conditions Stormy,2323,29.85,43.35
3,High,conditions Stormy,733,27.85,38.74
2,High,conditions Sandstorms,701,27.71,38.37
0,High,conditions Cloudy,744,28.94,37.63


### Section 5 Interpretation

**1. Which weather and traffic combinations create the highest delivery risk?**  
**Jam traffic combined with Fog or Cloudy weather** creates the highest risk. Jam + Fog has a **76.08% late-delivery rate** with an average delivery time of **36.81 minutes**, while Jam + Cloudy has a **74.84% late rate** and an average of **36.69 minutes**.

**2. How much does the combination of poor weather and heavy traffic increase delays?**  
Jam + Fog deliveries take approximately **10.52 minutes longer** than the overall average delivery time of 26.29 minutes. Jam + Cloudy deliveries take approximately **10.40 minutes longer**.

**3. Which combinations should trigger proactive operational intervention?**  
**Jam + Fog** and **Jam + Cloudy** should receive the highest priority because they affect a substantial number of orders (**2,429 and 2,349**) while producing late-delivery rates above **74%**.

**4. What actions can reduce delays under these conditions?**  
The business should use **dynamic ETA adjustments, route optimization, additional delivery capacity, and reduced multiple-order assignments** when heavy traffic coincides with Fog or Cloudy weather.

### Business Insight

The combination of **traffic and weather creates significantly higher delivery risk than either factor alone**. Jam + Fog and Jam + Cloudy conditions are the most critical operating scenarios, with average delivery times exceeding **36 minutes** and late-delivery rates above **74%**.

**Recommendation:** Use traffic and weather data together when predicting ETAs and allocating delivery workload. High-risk combinations should trigger proactive operational measures before delays occur.

## 6. Delivery Partner & Vehicle Performance Recommendation

Analyze delivery partner and vehicle performance to identify operational strengths and determine where performance improvements may be required.

### Key Business Questions

- Which delivery partners demonstrate the best and worst delivery performance?
- Which vehicle types have the highest delivery times and late-delivery rates?
- Should vehicle type or partner performance influence delivery assignment?
- What operational improvements can be recommended?

In [16]:
partner_recommendation = (
    df.groupby("Delivery_person_ID")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
)

# Focus on partners with sufficient delivery volume
partner_recommendation = (
    partner_recommendation[
        partner_recommendation["Orders"] >= 50
    ]
    .sort_values("Avg_Delivery_Time")
)

partner_recommendation["Avg_Delivery_Time"] = (
    partner_recommendation["Avg_Delivery_Time"].round(2)
)

partner_recommendation["Late_Rate"] = (
    partner_recommendation["Late_Rate"].round(2)
)

partner_recommendation.head(10)

,Delivery_person_ID,Orders,Avg_Delivery_Time,Late_Rate
1102,PUNERES07DEL02,59,22.71,11.86
994,MUMRES12DEL02,56,22.80,12.50
223,BANGRES15DEL02,59,23.07,25.42
592,HYDRES18DEL02,60,23.30,15.00
325,CHENRES08DEL02,54,23.59,18.52
310,CHENRES03DEL02,52,23.65,19.23
607,INDORES02DEL02,57,23.75,19.30
1245,SURRES16DEL01,62,23.81,20.97
1009,MUMRES17DEL02,56,23.84,23.21
364,COIMBRES01DEL02,55,24.00,23.64


In [17]:
partner_recommendation.sort_values(
    "Avg_Delivery_Time",
    ascending=False
).head(10)

,Delivery_person_ID,Orders,Avg_Delivery_Time,Late_Rate
309,CHENRES03DEL01,56,30.57,50.00
339,CHENRES14DEL01,52,30.10,55.77
1119,PUNERES14DEL01,56,29.41,41.07
1291,VADRES11DEL02,66,29.24,42.42
660,JAPRES010DEL01,55,29.15,49.09
406,COIMBRES16DEL02,58,29.09,39.66
318,CHENRES06DEL01,56,29.02,37.50
408,COIMBRES17DEL01,56,29.00,37.50
1263,VADRES01DEL01,61,28.97,40.98
689,JAPRES09DEL03,61,28.95,42.62


In [18]:
vehicle_recommendation = (
    df.groupby("Type_of_vehicle")
    .agg(
        Orders=("Time_taken(min)", "count"),
        Avg_Delivery_Time=("Time_taken(min)", "mean"),
        Late_Rate=("Late_Delivery", lambda x: (x == "Late").mean() * 100)
    )
    .reset_index()
    .sort_values("Late_Rate", ascending=False)
)

vehicle_recommendation["Avg_Delivery_Time"] = (
    vehicle_recommendation["Avg_Delivery_Time"].round(2)
)

vehicle_recommendation["Late_Rate"] = (
    vehicle_recommendation["Late_Rate"].round(2)
)

vehicle_recommendation

,Type_of_vehicle,Orders,Avg_Delivery_Time,Late_Rate
2,motorcycle,26435,27.61,33.94
0,bicycle,68,26.43,30.88
3,scooter,15276,24.48,24.34
1,electric_scooter,3814,24.47,23.44


### Section 6 Interpretation

**1. Which delivery partners demonstrate the best and worst delivery performance?**  
Among partners with at least **50 deliveries**, **PUNERES07DEL02** demonstrates the best average delivery performance at **22.71 minutes** with an **11.86% late-delivery rate**. The slowest partner is **CHENRES03DEL01**, with an average delivery time of **30.57 minutes** and a **50.00% late-delivery rate**. This represents a difference of approximately **7.86 minutes** in average delivery time.

**2. Which vehicle types have the highest delivery times and late-delivery rates?**  
**Motorcycles** have the highest average delivery time at **27.61 minutes** and the highest late-delivery rate at **33.94%**. Scooters and electric scooters perform better, with late rates of **24.34%** and **23.44%**, respectively. Bicycle performance should be interpreted cautiously because there are only **68 orders**.

**3. Should vehicle type or partner performance influence delivery assignment?**  
Yes. Delivery partner performance and vehicle type can be considered when assigning deliveries, particularly for difficult orders. However, assignments should also consider traffic, weather, distance, and workload rather than relying on a single factor.

**4. What operational improvements can be recommended?**  
The business should identify consistently high-performing delivery partners and use their practices as a benchmark. Partners with consistently higher delivery times should receive targeted performance support. Vehicle allocation should also consider operating conditions, with scooters and electric scooters potentially preferred for suitable deliveries.

### Business Insight

Delivery performance varies substantially across both **delivery partners and vehicle types**. The best-performing partner averages **22.71 minutes**, compared with **30.57 minutes** for the slowest high-volume partner. Motorcycles also show the highest late-delivery rate among the major vehicle categories.

**Recommendation:** Use delivery partner performance and vehicle type as inputs to operational assignment and performance management, while combining them with traffic, weather, distance, and workload conditions for better delivery decisions.

## 7. Final Business Recommendations

Consolidate the major analytical findings into actionable recommendations that can improve delivery performance and reduce late deliveries.

### Key Business Questions

- Which operational factors should receive the highest priority?
- What actions can reduce the overall late-delivery rate?
- Which delivery situations require proactive intervention?
- What overall strategy should the business adopt to improve delivery performance?

In [19]:
final_recommendations = pd.DataFrame({
    "Priority": [
        "High",
        "High",
        "High",
        "Medium",
        "Medium",
        "Medium"
    ],
    "Operational_Area": [
        "Traffic Management",
        "Weather & Traffic",
        "Delivery Workload",
        "Vehicle Allocation",
        "Delivery Partner Performance",
        "ETA Planning"
    ],
    "Key_Finding": [
        "Jam traffic has a 50.64% late-delivery rate.",
        "Jam + Fog reaches a 76.08% late-delivery rate.",
        "2+ deliveries show a 100% late-delivery rate.",
        "Motorcycles have a 33.94% late-delivery rate.",
        "Partner performance varies substantially.",
        "Overall late-delivery rate is 29.84%."
    ],
    "Recommended_Action": [
        "Use traffic-aware routing and ETA adjustments.",
        "Apply proactive intervention during high-risk weather and traffic combinations.",
        "Limit multiple-order assignments during difficult operating conditions.",
        "Match suitable vehicles to traffic, distance, and workload conditions.",
        "Monitor partner performance and provide targeted support.",
        "Use operational factors to improve ETA prediction."
    ]
})

final_recommendations

,Priority,Operational_Area,Key_Finding,Recommended_Action
0,High,Traffic Management,Jam traffic has a 50.64% late-delivery rate.,Use traffic-aware routing and ETA adjustments.
1,High,Weather & Traffic,Jam + Fog reaches a 76.08% late-delivery rate.,Apply proactive intervention during high-risk ...
2,High,Delivery Workload,2+ deliveries show a 100% late-delivery rate.,Limit multiple-order assignments during diffic...
3,Medium,Vehicle Allocation,Motorcycles have a 33.94% late-delivery rate.,"Match suitable vehicles to traffic, distance, ..."
4,Medium,Delivery Partner Performance,Partner performance varies substantially.,Monitor partner performance and provide target...
5,Medium,ETA Planning,Overall late-delivery rate is 29.84%.,Use operational factors to improve ETA predict...


### Section 7 Interpretation

**1. Which operational factors should receive the highest priority?**  
**Traffic, weather × traffic interactions, and delivery workload** should receive the highest priority. Jam traffic has a **50.64% late-delivery rate**, while Jam + Fog reaches **76.08%**, and deliveries involving 2+ orders have historically shown a **100% late-delivery rate**.

**2. What actions can reduce the overall late-delivery rate?**  
The business should use **traffic-aware routing, dynamic ETA adjustments, proactive weather-based planning, and workload controls** to reduce delivery delays.

**3. Which delivery situations require proactive intervention?**  
The highest priority should be given to deliveries involving **Jam traffic combined with Fog or Cloudy weather**, particularly when the delivery partner has multiple orders.

**4. What overall strategy should the business adopt to improve delivery performance?**  
The business should move toward a **data-driven delivery management strategy** that combines traffic, weather, workload, vehicle type, delivery partner performance, and ETA planning when making operational decisions.

### Business Insight

The analysis shows that delivery delays are driven by a **combination of operational factors rather than a single variable**. Traffic and workload are particularly important because they affect both delivery time and late-delivery risk.

**Final Recommendation:** Build a proactive delivery management system that uses **traffic and weather conditions to adjust ETAs, controls multiple-order assignments during high-risk conditions, considers vehicle suitability, and continuously monitors delivery partner performance**. This approach can help reduce the current **29.84% late-delivery rate** while improving overall operational efficiency.